# 2.2_huim — Candidate-aware attention for relevance shortlisting

This notebook runs one final workflow: **Optuna → apply the winning parameters → train on all recorded training requests → restore the selected validation checkpoint → export all requests → save and reload from shared Drive**.

Stage 1 scores every time-eligible article in the shared catalogue and retains its genuine **top 200**. Stage 2 applies the business priorities and displays five recommendations. Selection remains **known-click Recall@100**, then observed-click nDCG@100, then the earlier checkpoint/trial.

Completed run: `20261104_014752_seed42`. Section 9 records the validation
findings and limitations.


## 1. Business hypothesis and experiment overview

| Item | Definition |
| --- | --- |
| Experiment / owner | `2.2_huim` / Winston Ho Rui Wei |
| Business objective | Relevant recommendations that support continued reading and reader retention (Interim Report §1.1, §1.2.3) |
| Hypothesis | Readers have several interests. Matching each candidate to individual past clicks should recover more of their known interests than one history average. |
| Model | HUIM-inspired candidate-conditioned attention over fixed history events, article metadata and frozen text/image embeddings; a separate request/pair feature branch and MLP produce click logits. |
| Evidence | Development validation Recall@100 and observed-click nDCG@100 on the declared eligible pool; recorded-slate AUC is a diagnostic. Later shared evaluation will examine representative recommendations and compare models on identical requests. |

The article tower projects frozen 768-dimensional text and 1,024-dimensional image vectors and combines them with metadata embeddings and missingness flags. Each history event also carries train-fitted reading-depth bins and a recency bin. Attention uses `[history, candidate, difference, product]`, a small MLP and masked softmax. The model has no recurrence or positional encoding: recency belongs to each event, and permuting aligned events does not change a prediction. The final feature fusion uses concatenation and LayerNorm.

This adapts HUIM's target attention and feature interactions. It does not reproduce its whole-catalogue PCA, slate-dependent popularity features, history augmentation or reported paper scores. Fixed histories contain events strictly before the split's first request. Training uses BCE on the supplied recorded candidates: shown-clicked = 1, shown-unclicked = 0. Unshown catalogue articles have unknown outcomes and never become training negatives.

**Selection:** macro known-click Recall@100 over the same 100 validation requests, each scored against the full eligible catalogue; observed-click nDCG@100 breaks ties and exact ties retain the earlier checkpoint/trial. All original clicks remain in recall denominators and ideal DCG, including clicks rejected by eligibility. These are development results because validation is reused for selection. Project holdout targets are reserved for final evaluation after pipeline freeze.

Course connection: the implementation combines categorical embeddings, attention, feature interactions, dense networks and dropout/weight-decay regularisation. The complete HUIM-inspired architecture is an adaptation rather than a model taught as a whole in class.


## 2. Setup and run configuration

Use an **A100 GPU / High RAM** Colab runtime. Upload `huim_helpers.zip` beside
this notebook to `/content`. Locally, keep the same ZIP beside the notebook.
It contains the editable `huim/` sources and `huim_freshness.py` fast ranker.

Add the shared folder shortcut under `MyDrive/Colab Notebooks/BT4222 Shared Final Folder`. Inputs stay there; the complete run is saved under `Experiments/2.2_huim/<run_id>/` in that folder.


In [ ]:
from pathlib import Path
import time
from datetime import datetime
from zoneinfo import ZoneInfo

SEED = 42
OWNER = "Winston Ho Rui Wei"
DATA_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder")
OUTPUT_ROOT = DATA_ROOT / "Experiments" / "2.2_huim"
LOCAL_ROOT = Path("/content/huim_runs")
RUN_ID = f"{datetime.now(ZoneInfo('Asia/Singapore')):%Y%m%d_%H%M%S}_seed{SEED}"
TUNING_TRIALS = 12
TUNING_MAX_EPOCHS = 2
TUNING_TRAIN_REQUESTS = 60_000


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("Local execution: set DATA_ROOT and LOCAL_ROOT above to your local folders.")
assert DATA_ROOT.is_dir(), f"Shared data folder not found: {DATA_ROOT}"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)

import importlib, subprocess, sys
for module, package in {"numpy": "numpy>=1.24", "pandas": "pandas>=2.0", "pyarrow": "pyarrow>=14",
                        "torch": "torch>=2.3", "optuna": "optuna>=3.6,<5", "duckdb": "duckdb>=1.0"}.items():
    if importlib.util.find_spec(module) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)


Mounted at /content/drive


In [ ]:
import zipfile, hashlib, json, shutil
roots = [Path("/content"), Path.cwd()]
EXP_DIR = next((root for root in roots if (root / "huim_helpers.zip").is_file()), None)
assert EXP_DIR is not None, "Upload huim_helpers.zip beside the notebook."
with zipfile.ZipFile(EXP_DIR / "huim_helpers.zip") as archive:
    archive.extractall(EXP_DIR)
for name in [name for name in sys.modules if name == "huim" or name.startswith("huim.")]:
    del sys.modules[name]
sys.path.insert(0, str(EXP_DIR))
import huim
import huim_freshness as H
from huim import pipeline as P, catalogue as C, features as F, data as D
from huim.io_utils import RunLogger, atomic_write_json, environment_info, read_json
from huim.model import parameter_counts
import numpy as np, pandas as pd, torch, optuna
SPLIT_MANIFEST = json.loads((DATA_ROOT / "split_manifest.json").read_text())
if SPLIT_MANIFEST.get("protocol") != "ebnerd_small_labelled_user_holdout_v1":
    raise ValueError("Expected the labelled project dataset and its split manifest.")
if json.loads((DATA_ROOT / "feature_engineering" / "split_manifest.json").read_text()) != SPLIT_MANIFEST:
    raise ValueError("Feature outputs and dataset split manifest do not match.")


HELPER_HASHES = {path.name: hashlib.sha256(path.read_bytes()).hexdigest()
                 for path in sorted((EXP_DIR / "huim").glob("*.py"))}
CODE_SOURCE = {"source": "uploaded huim_helpers.zip or adjacent huim folder", "commit": None,
               "uncommitted_changes": None, "helper_sha256": HELPER_HASHES,
               "note": "Exact executed helper code is copied to model/code/huim; no Git revision is inferred."}
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CFG = P.make_config(seed=SEED, max_epochs=4, train_requests=None, selection_requests=100,
                    export_requests={split: None for split in D.SPLITS}, selection_interval_steps=300)
RUN_DIR, PERSIST_DIR = LOCAL_ROOT / RUN_ID, OUTPUT_ROOT / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
atomic_write_json(RUN_DIR / "split_manifest.json", SPLIT_MANIFEST)
log = RunLogger(RUN_DIR)
log(f"run {RUN_ID}; device {DEVICE}; all training/export requests; selection=100; export K=200")
print("Training AMP:", CFG["amp"] and DEVICE.type == "cuda")
print("Tuning:", TUNING_TRIALS, "trials ×", TUNING_MAX_EPOCHS, "epochs, up to", TUNING_TRAIN_REQUESTS, "complete requests/trial")


2026-11-03 17:48:27 run 20261104_014752_seed42; device cuda; all training/export requests; selection=100; export K=200
Training AMP: True
Tuning: 12 trials × 2 epochs, up to 60000 complete requests/trial


## 3. Load and check shared data

Load the supplied recorded pair keys/labels, behaviours and fixed histories. Build **one ordered article union: train_val, then test; first source record wins, including missing fields**. The same union and vector precedence apply to every split. An article is eligible only when its known publication time is at or before the request time. Missing embeddings keep the article eligible under the model's documented missing-vector fallback.

All training and export requests are retained. The fixed 100 validation-selection requests use the shared seed-42 BLAKE2b request rule used by 3.1. Each selected request uses its complete eligible article pool. Input preparation decodes each vector source once and records reader, dimensions, missingness and source precedence.


In [ ]:
with log.stage("prepare shared inputs"):
    ws = P.prepare_inputs(CFG, DATA_ROOT, log=log)
    INPUT_MANIFEST = D.input_manifest(ws.paths)
    SCOPES = P.resolve_request_scopes(ws, CFG)
    atomic_write_json(RUN_DIR / "input_manifest.json", INPUT_MANIFEST)
    atomic_write_json(RUN_DIR / "selection" / "request_scopes.json",
                      {"pool_fingerprint": ws.reports["pool_fingerprint"], "pool_policy": ws.reports["pool_policy"],
                       **P.scope_ids(ws, SCOPES)})
    PREFLIGHT = P.preflight(ws, CFG, [LOCAL_ROOT, OUTPUT_ROOT], SCOPES)
    atomic_write_json(RUN_DIR / "preflight.json", PREFLIGHT)
BUDGET = PREFLIGHT["budget"]
display(pd.DataFrame({split: {"requests": len(ws.ctx[split].requests),
                              "recorded_pairs": len(ws.pairs[split]),
                              "catalogue_articles": ws.ctx[split].report["catalogue_articles"],
                              "eligible_pairs_to_score": BUDGET[split]["planned_export"]["eligible_pairs"],
                              "retained_score_rows": BUDGET[split]["planned_export"]["retained_score_rows"],
                              "retained_output_GB_estimate": BUDGET[split]["planned_export"]["retained_output_bytes_estimate"] / 1e9}
                      for split in D.SPLITS}).T)
print("Pool:", ws.reports["pool_fingerprint"])
print("Embedding coverage:", PREFLIGHT["coverage"])
print("Recorded candidates rejected by catalogue eligibility:", PREFLIGHT["recorded_ineligible"])
print("Selection pairs per evaluation:", BUDGET["selection_per_evaluation_pairs"])
print("Output estimates use uncompressed retained row/list sizes; actual Parquet sizes depend on compression.")


2026-11-03 17:48:27 [start] prepare shared inputs
2026-11-03 17:49:03 articles: 125,541 unique IDs from ['train_val', 'test']
2026-11-03 17:49:16 vectors: text 20,997, image 17,243
2026-11-03 17:49:44 recorded pairs: train 2,585,747, validation 344,666, test 348,951
2026-11-03 17:49:59 inputs ready in 92s
2026-11-03 17:50:07 [done] prepare shared inputs (99.9s)


,requests,recorded_pairs,catalogue_articles,eligible_pairs_to_score,retained_score_rows,retained_output_GB_estimate
train,232887.0,2585747.0,125541.0,2.855294e+10,46577400.0,2.615787
validation,29110.0,344666.0,125541.0,3.608175e+09,5822000.0,0.326964
test,29142.0,348951.0,125541.0,3.611848e+09,5828400.0,0.327323


Pool: {'policy': 'published_at_or_before_request_v1', 'articles': 125541, 'ids_and_published_sha256': 'cb908695ad543fdce3fe08042a650e66f7b73f972310debc7c21b3191f5e927f', 'sources': {}}
Embedding coverage: {'train': {'catalogue_articles': 125541, 'catalogue_with_text': 20997, 'catalogue_with_image': 17243, 'catalogue_text_share': 0.16725213276937415, 'catalogue_image_share': 0.1373495511426546, 'recorded_candidates_text_share': 1.0, 'recorded_candidates_image_share': 0.716792091414976, 'recorded_candidates_unknown_article': 0}, 'validation': {'catalogue_articles': 125541, 'catalogue_with_text': 20997, 'catalogue_with_image': 17243, 'catalogue_text_share': 0.16725213276937415, 'catalogue_image_share': 0.1373495511426546, 'recorded_candidates_text_share': 1.0, 'recorded_candidates_image_share': 0.6966135331016114, 'recorded_candidates_unknown_article': 0}, 'test': {'catalogue_articles': 125541, 'catalogue_with_text': 20997, 'catalogue_with_image': 17243, 'catalogue_text_share': 0.16725213

## 4. Prepare model inputs

Request/article/history features are constructed for arbitrary pairs with one shared inference path. Vocabularies, engagement bins, body scaling and dense standardisation are fitted on permitted training data. Publication cutoff, category preferences and missingness indicators use only request-time context; slate size, in-slate rank, current reading outcomes and later behaviour clicks are excluded.

The saved feature dictionary preserves ordered feature names and category index-to-value maps. `preprocessing.json` preserves the fitted transforms and `article_ids.npy` the original article-ID order. Embedding coordinates are representation dimensions, not word-level explanations.


In [ ]:
with log.stage("fit training preprocessing"):
    F.fit_dense_scalers(ws.pre, ws.ctx["train"], ws.art)
FEATURE_ORDER = {"request_context": F.REQUEST_DENSE_COLUMNS(ws.pre), "candidate": F.PAIR_DENSE_COLUMNS}
display(pd.DataFrame(F.FEATURE_AUDIT, columns=["feature", "scope", "status", "reason"]))
print("Named dense inputs:", FEATURE_ORDER)


2026-11-03 17:50:07 [start] fit training preprocessing
2026-11-03 17:50:08 [done] fit training preprocessing (0.9s)


,feature,scope,status,reason
0,category / subcategory / article_type / sentim...,article,used,static catalogue metadata; available for any c...
1,body_word_count,article,used,from the supplied snapshot (may post-date edit...
2,"text vector (gemini-embedding-2, 768-d, frozen)",article,used with mask,precomputed lookup; missing vectors kept missi...
3,"image vector (EB-NeRD supplied, 1024-d, frozen)",article,used with mask,separate space from text; missing kept missing...
4,article age = request time - published_time,pair,used,known at request; future articles are ineligible
5,premium x subscriber access,pair,used,request subscriber flag x article premium
6,history category/type/topic preference crosses,pair,used,computed from fixed history before the split s...
7,candidate previously clicked in fixed history,pair,used,fixed history only
8,device / SSO / subscriber / hour / weekday,request,used,request context
9,"history size, median read time, mean scroll, 7...",request,used,fixed-history aggregates; past engagement only


Named dense inputs: {'request_context': ['device_1', 'device_2', 'device_3', 'device_missing', 'device_unknown', 'is_sso_user', 'sso_missing', 'is_subscriber', 'subscriber_missing', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'log_history_size', 'history_empty', 'log_median_read_time', 'read_time_missing', 'mean_scroll_frac', 'scroll_missing', 'log_history_clicks_7d', 'log_hours_since_last_click', 'last_click_missing'], 'candidate': ['log_article_age_h', 'article_age_missing', 'access_premium_subscriber', 'access_premium_non_subscriber', 'access_standard_subscriber', 'access_standard_non_subscriber', 'access_missing', 'category_preference', 'category_preference_missing', 'article_type_preference', 'article_type_preference_missing', 'topic_preference', 'topic_preference_missing', 'previously_clicked']}


## 5. Define and train model

**Mandatory bounded search:** 12 seeded TPE trials, at most two epochs and 60,000 complete training requests per trial. Search learning rate, weight decay, dropout, article dimension and modality dropout. Every trial uses the same model-training seed (42), the same training request prefix and the fixed validation-selection requests. Median pruning uses intermediate catalogue Recall@100. Save every trial outcome and a consistent SQLite study snapshot to shared Drive.

Apply the winner before the main fit on **all recorded training requests**. AdamW, GPU mixed precision and gradient clipping optimise BCE. Evaluate Recall@100 every 300 steps and at each epoch end; the scheduler and early stopping follow that objective. The main budget is four epochs, subject to early stopping. Restore and re-evaluate the selected checkpoint before exporting.


In [ ]:
from huim.tune import run_study, select_best_trial, SEARCH_SPACE
SELECTOR = P.make_selector(ws, CFG, DEVICE, SCOPES)
def objective(trial, params):
    trial_cfg = dict(CFG, **params, max_epochs=TUNING_MAX_EPOCHS, train_requests=TUNING_TRAIN_REQUESTS,
                     checkpoint_every_steps=0)
    def report(record):
        trial.report(record["known_click_recall@100"], record["eval_index"])
        if trial.should_prune():
            raise optuna.TrialPruned()
    _, _, state, subset, _ = P.train_model(
        ws, trial_cfg, RUN_DIR / "tuning" / f"trial_{trial.number:03d}", None, DEVICE,
        resume=False, log=log, selector=SELECTOR, on_evaluation=report)
    best = state["best"]
    for name in ("observed_click_ndcg@100", "diagnostic_recorded_slate_auc", "epoch", "global_step"):
        trial.set_user_attr(name, best[name])
    trial.set_user_attr("training_scope", subset)
    return best["known_click_recall@100"]

with log.stage("Optuna validation search"):
    study = run_study(objective, f"huim_{RUN_ID}", RUN_DIR / "tuning", PERSIST_DIR, TUNING_TRIALS, SEED)
winner = select_best_trial(study)
assert winner is not None, "No trial completed; inspect saved tuning/trials.csv."
CFG.update(winner["params"])
STUDY_SUMMARY = {"search_space": SEARCH_SPACE, "actual_trials": len(study.trials),
                 "states": pd.Series([trial.state.name for trial in study.trials]).value_counts().to_dict(),
                 "selected_trial": winner, "applied_to_main_training": True,
                 "sampler": "TPESampler(seed=42)",
                 "pruner": "MedianPruner(n_startup_trials=3, n_warmup_steps=1)",
                 "objective": "Recall@100, then observed-click nDCG@100, then earlier trial",
                 "model_seed_every_trial": SEED, "selection_sample": SELECTOR.protocol["sample"],
                 "budget": {"trials": TUNING_TRIALS, "max_epochs": TUNING_MAX_EPOCHS,
                            "training_requests_per_trial": TUNING_TRAIN_REQUESTS}}
atomic_write_json(RUN_DIR / "tuning" / "summary.json", STUDY_SUMMARY)
atomic_write_json(RUN_DIR / "tuning" / "tuned_config.json", CFG)
for name in ("summary.json", "tuned_config.json"):
    shutil.copyfile(RUN_DIR / "tuning" / name, PERSIST_DIR / "tuning" / name)
display(study.trials_dataframe())
print("Applied winning parameters:", winner["params"])


2026-11-03 17:50:08 [start] Optuna validation search


[I 2026-11-03 17:50:10,510] A new study created in RDB with name: huim_20261104_014752_seed42
/content/huim/train.py:224: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total += float(loss) * len(batch)


2026-11-03 17:50:32 eval 0 (epoch 0 step 164): Recall@100 0.65500  nDCG@100 0.22552  [diag AUC 0.67228]  *selected*
2026-11-03 17:50:46 eval 1 (epoch 1 step 136): Recall@100 0.64500  nDCG@100 0.18021  [diag AUC 0.68224]
2026-11-03 17:50:57 eval 2 (epoch 1 step 164): Recall@100 0.62500  nDCG@100 0.18267  [diag AUC 0.68382]


[I 2026-11-03 17:51:06,342] Trial 0 finished with value: 0.655 and parameters: {'lr': 0.0007106591851092234, 'dropout': 0.38521429192297485, 'weight_decay': 0.000157029708840554, 'd_model': 48, 'modality_dropout': 0.017425083650459836}. Best is trial 0 with value: 0.655.


2026-11-03 17:51:23 eval 0 (epoch 0 step 164): Recall@100 0.69000  nDCG@100 0.22485  [diag AUC 0.68544]  *selected*
2026-11-03 17:51:38 eval 1 (epoch 1 step 136): Recall@100 0.69000  nDCG@100 0.22315  [diag AUC 0.69950]
2026-11-03 17:51:50 eval 2 (epoch 1 step 164): Recall@100 0.72000  nDCG@100 0.22013  [diag AUC 0.70316]  *selected*


[I 2026-11-03 17:52:01,340] Trial 1 finished with value: 0.72 and parameters: {'lr': 0.0022044355215811344, 'dropout': 0.2803345035229627, 'weight_decay': 0.000133112160807369, 'd_model': 64, 'modality_dropout': 0.06370173320348284}. Best is trial 1 with value: 0.72.


... (19 outputs left out)

[I 2026-11-03 17:59:15,163] Trial 11 pruned. 


2026-11-03 17:59:15 [done] Optuna validation search (546.6s)


,number,value,datetime_start,datetime_complete,duration,params_d_model,params_dropout,params_lr,params_modality_dropout,params_weight_decay,user_attrs_diagnostic_recorded_slate_auc,user_attrs_epoch,user_attrs_global_step,user_attrs_observed_click_ndcg@100,user_attrs_training_scope,state
0,0,0.655,2026-11-03 17:50:10.520533,2026-11-03 17:51:06.327711,0 days 00:00:55.807178,48,0.385214,0.000711,0.017425,0.000157,0.672277,0.0,164.0,0.225515,"{'purpose': 'training', 'split': 'train', 'lim...",COMPLETE
1,1,0.720,2026-11-03 17:51:06.403415,2026-11-03 17:52:01.328701,0 days 00:00:54.925286,64,0.280335,0.002204,0.063702,0.000133,0.703156,1.0,328.0,0.220135,"{'purpose': 'training', 'split': 'train', 'lim...",COMPLETE
2,2,0.665,2026-11-03 17:52:01.400565,2026-11-03 17:52:50.551310,0 days 00:00:49.150745,48,0.155021,0.000456,0.183556,0.000008,0.670071,0.0,164.0,0.229984,"{'purpose': 'training', 'split': 'train', 'lim...",COMPLETE
3,3,0.605,2026-11-03 17:52:50.610816,2026-11-03 17:53:23.238649,0 days 00:00:32.627833,64,0.187643,0.000414,0.154270,0.000013,NaN,NaN,NaN,NaN,NaN,PRUNED
4,4,0.735,2026-11-03 17:53:23.298057,2026-11-03 17:54:31.703738,0 days 00:01:08.405681,96,0.113935,0.001174,0.289690,0.000066,0.695610,1.0,300.0,0.219942,"{'purpose': 'training', 'split': 'train', 'lim...",COMPLETE
5,5,0.795,2026-11-03 17:54:31.775214,2026-11-03 17:55:21.171667,0 days 00:00:49.396453,48,0.191384,0.001930,0.148553,0.000002,0.700808,1.0,300.0,0.233414,"{'purpose': 'training', 'split': 'train', 'lim...",COMPLETE
6,6,0.520,2026-11-03 17:55:21.234735,2026-11-03 17:55:50.738851,0 days 00:00:29.504116,48,0.372796,0.000325,0.164013,0.000006,NaN,NaN,NaN,NaN,NaN,PRUNED
7,7,0.575,2026-11-03 17:55:50.798501,2026-11-03 17:56:20.397001,0 days 00:00:29.598500,48,0.390875,0.000459,0.276562,0.000212,NaN,NaN,NaN,NaN,NaN,PRUNED
8,8,0.615,2026-11-03 17:56:20.453869,2026-11-03 17:56:53.281084,0 days 00:00:32.827215,64,0.158795,0.000368,0.248621,0.000001,NaN,NaN,NaN,NaN,NaN,PRUNED
9,9,0.665,2026-11-03 17:56:53.347413,2026-11-03 17:57:26.074596,0 days 00:00:32.727183,64,0.184280,0.000682,0.296066,0.000042,NaN,NaN,NaN,NaN,NaN,PRUNED


Applied winning parameters: {'lr': 0.0019298279978549665, 'dropout': 0.19138413075201122, 'weight_decay': 1.9634341572933354e-06, 'd_model': 48, 'modality_dropout': 0.14855307303338106}


In [ ]:
with log.stage("fit all recorded training requests"):
    model, MODEL_CFG, TRAIN_STATE, TRAIN_SUBSET, SELECTION = P.train_model(
        ws, CFG, RUN_DIR, PERSIST_DIR, DEVICE, resume=False, log=log, selector=SELECTOR)
assert TRAIN_SUBSET["requests"] == len(ws.ctx["train"].requests)
assert SELECTION["restored_matches_recorded"], "Selected checkpoint re-evaluation disagrees with its recorded result."
# Preserve the selected model and every fitted transform before catalogue export.
shutil.copytree(RUN_DIR / "model", PERSIST_DIR / "model", dirs_exist_ok=True)
shutil.copytree(RUN_DIR / "selection", PERSIST_DIR / "selection", dirs_exist_ok=True)
shutil.copyfile(RUN_DIR / "training_history.json", PERSIST_DIR / "training_history.json")
PARAMS = parameter_counts(model)
print("Actual stop reason:", TRAIN_STATE["stop_reason"])
print("Selected checkpoint:", SELECTION["selected_checkpoint"])
print("Parameters:", PARAMS)


2026-11-03 17:59:15 [start] fit all recorded training requests
2026-11-03 17:59:22 epoch 0 step 200/632 loss 0.29389 (0.029s/step)
2026-11-03 17:59:34 eval 0 (epoch 0 step 300): Recall@100 0.64500  nDCG@100 0.19010  [diag AUC 0.69716]  *selected*
2026-11-03 17:59:37 epoch 0 step 400/632 loss 0.27829 (0.029s/step)
2026-11-03 17:59:43 epoch 0 step 600/632 loss 0.27593 (0.029s/step)
2026-11-03 17:59:53 eval 1 (epoch 0 step 600): Recall@100 0.72500  nDCG@100 0.21239  [diag AUC 0.70960]  *selected*
2026-11-03 18:00:04 eval 2 (epoch 0 step 632): Recall@100 0.73500  nDCG@100 0.24101  [diag AUC 0.70783]  *selected*
2026-11-03 18:00:10 epoch 1 step 200/632 loss 0.27385 (0.029s/step)
2026-11-03 18:00:22 eval 3 (epoch 1 step 268): Recall@100 0.80500  nDCG@100 0.22326  [diag AUC 0.71429]  *selected*
2026-11-03 18:00:26 epoch 1 step 400/632 loss 0.27302 (0.029s/step)
2026-11-03 18:00:41 eval 4 (epoch 1 step 568): Recall@100 0.81500  nDCG@100 0.22919  [diag AUC 0.71670]  *selected*
2026-11-03 18:00:

## 6. Validation results

Report the selected checkpoint on the fixed 100-request selection sample and recorded-slate AUC separately. All-validation Recall@100 and Recall@200 follow from the export below. Saved training histories and Optuna trials support later learning curves and search analysis. Detailed attention examples, robustness checks, feature analysis and cross-model plots belong in the planned shared evaluation notebook.


In [ ]:
with log.stage("selected validation results"):
    SELECTED_SAMPLE = SELECTOR.evaluate(model)
    art_d, contexts_d = P.device_and_tensors(ws, DEVICE)
    VAL_DIAGNOSTIC, _ = P.evaluate_recorded(model, contexts_d["validation"], art_d, DEVICE)
display(pd.Series({key: SELECTED_SAMPLE[key] for key in (
    "known_click_recall@100", "observed_click_ndcg@100", "eligible_only_recall@100",
    "requests_evaluated", "clicked_articles_total", "clicked_articles_eligible")}))
print("Recorded-slate diagnostic:", VAL_DIAGNOSTIC)


2026-11-03 18:02:39 [start] selected validation results
2026-11-03 18:02:48 [done] selected validation results (9.7s)


,0
known_click_recall@100,0.825000
observed_click_ndcg@100,0.246918
eligible_only_recall@100,0.825000
requests_evaluated,100.000000
clicked_articles_total,101.000000
clicked_articles_eligible,101.000000


Recorded-slate diagnostic: {'mean_impression_auc': 0.7174821727091404, 'mean_impression_ndcg@5': 0.5501385324050402, 'candidate_rows': 344666, 'impressions': 29110, 'auc_eligible_impressions': 29110, 'auc_excluded_single_class': 0, 'ndcg_eligible_impressions': 29110, 'ndcg_excluded_no_positive': 0}


## 7. Predict and export all splits

For every train, validation and test request, score the **complete eligible union catalogue** in bounded chunks. Keep the genuine top `min(200, eligible_count)`, ordered by descending logit then ascending original article ID. A running top-K retains only 201 items per request while checking each merge, including ties at 100 and 200. Never insert recorded candidates or known clicks. Empty pools remain explicit as `[]`.

| Artifact | Contents |
| --- | --- |
| `predictions/{split}/submission_article_scores.parquet` | Retained top-200 pairs: string behaviour ID, int64 article ID, 1-based rank, float32 logit and its uncalibrated sigmoid probability |
| `rankings/{split}/submission_top_200.parquet` | Every request, native list<int64> in the same order |
| `diagnostics/recorded_slates/{split}_scores.parquet` | Every original shown candidate separately, including shown nonclicks and articles outside the shortlist; labels are joined separately |
| `diagnostics/catalogue/{split}_requests.parquet` | IDs, timestamps, eligible counts, empty pools and known-click retrieval bookkeeping |

No exhaustive catalogue matrix or score CSV is exported. Selection K stays 100; export K and shortlist size are 200; final display size is five. Unknown/unshown outcomes remain unknown, and holdout targets are not used or evaluated during development.


In [16]:
art_device, contexts = P.device_and_tensors(ws, DEVICE)
RECORDED, EXPORTS = {}, {}
for split in D.SPLITS:
    t0 = time.time()
    cpu, ctx = ws.ctx[split], contexts[split]
    requests = cpu.requests
    raw = P.score_recorded(model, ctx, art_device, DEVICE, amp=False)
    path = RUN_DIR/"diagnostics"/"recorded_slates"/f"{split}_scores.parquet"
    H.write_recorded(path, ctx, raw, H.recorded_age(ctx, art_device), 0.0)
    RECORDED[split] = {"path": str(path.relative_to(RUN_DIR)), "rows": len(raw),
                      "validation": {"passed": True, "rows": len(raw)}}
    print(split, "recorded-slate diagnostic rows", len(raw))
    if split == "validation":
        VAL_DIAGNOSTIC = P.impression_metrics(cpu.recorded["req_pos"],
            cpu.recorded["label"].astype(int), raw, k=5)
        RECORDED[split]["metrics"] = VAL_DIAGNOSTIC
    del raw
    scorer = H.FreshnessScorer(model, ctx, art_device, DEVICE, 131072)
    writer = H.ShortlistWriter(RUN_DIR, split)
    eligible = C.catalogue_eligibility(cpu, ws.art)["eligible_count"]
    parts, recent_parts = [], []
    next_progress = 1000
    for start in range(0, len(requests), 16):
        positions = np.arange(start, min(start+16, len(requests)))
        result = scorer.rank_many(positions, [0.0], 200)[0.0]
        writer.add(requests.iloc[positions], result)
        if split == "validation":
            _, rows = H.retrieval(requests, positions, result["article_ids"], result["lengths"])
            parts.append(rows)
            ids, lengths = scorer.newest(positions, 100)
            _, rows = H.retrieval(requests, positions, ids, lengths)
            recent_parts.append(rows)
        done = int(positions[-1])+1
        if done >= next_progress:
            elapsed = time.time()-t0
            print(split, done, "/", len(requests), "| elapsed min", round(elapsed/60, 1),
                  "| ETA min", round(elapsed/done*(len(requests)-done)/60, 1))
            next_progress += 1000
    files = writer.close()
    assert writer.n_requests == len(requests)
    assert writer.eligible_pairs == int(eligible.sum())
    assert writer.n_scores == int(np.minimum(200, eligible).sum())
    summary = {"requests": writer.n_requests, "scored_pairs": writer.eligible_pairs,
        "retained_score_rows": writer.n_scores, "empty_pools": int((eligible == 0).sum()),
        "seconds": time.time()-t0}
    if split == "validation":
        EXPORTED_VALIDATION = H.summarise_retrieval(pd.concat(parts, ignore_index=True))
        NEWEST_FIRST_METRICS = H.summarise_retrieval(pd.concat(recent_parts, ignore_index=True))
        summary["recall"] = EXPORTED_VALIDATION
    EXPORTS[split] = {"summary": summary, "validation": {"passed": True},
        "progress": {"scored_pairs": writer.eligible_pairs, "rows_scores": writer.n_scores,
                     "seconds": summary["seconds"]}}
    print(split, "complete:", summary)
    del scorer, parts, recent_parts
    torch.cuda.empty_cache()
shutil.copy2(EXP_DIR/"huim_freshness.py", RUN_DIR/"model"/"code"/"huim_freshness.py")
display(pd.DataFrame({s: {k: EXPORTS[s]["summary"][k] for k in
    ("requests", "scored_pairs", "retained_score_rows", "empty_pools", "seconds")}
    for s in ("validation", "test")}).T)
print("All-validation development retrieval metrics:", EXPORTED_VALIDATION)


2026-11-04 01:53:25 train: 16/232,887 requests; 1,951,696 eligible pairs scored
2026-11-04 01:55:47 train: 1,600/232,887 requests; 195,170,977 eligible pairs scored
2026-11-04 01:58:11 train: 3,200/232,887 requests; 390,347,108 eligible pairs scored
2026-11-04 02:00:36 train: 4,800/232,887 requests; 585,531,815 eligible pairs scored
2026-11-04 02:03:00 train: 6,400/232,887 requests; 780,725,719 eligible pairs scored
2026-11-04 02:05:25 train: 8,000/232,887 requests; 975,939,015 eligible pairs scored
2026-11-04 02:07:50 train: 9,600/232,887 requests; 1,171,170,449 eligible pairs scored
2026-11-04 02:10:14 train: 11,200/232,887 requests; 1,366,424,458 eligible pairs scored
2026-11-04 02:12:39 train: 12,800/232,887 requests; 1,561,690,686 eligible pairs scored
2026-11-04 02:15:04 train: 14,400/232,887 requests; 1,756,975,182 eligible pairs scored
2026-11-04 02:17:29 train: 16,000/232,887 requests; 1,952,289,763 eligible pairs scored
2026-11-04 02:19:54 train: 17,600/232,887 requests; 2,14

,requests,scored_pairs,retained_score_rows,empty_pools,seconds
validation,29110.0,566218581.0,5822000.0,0.0,118.409535
test,29142.0,566598922.0,5828400.0,0.0,113.832058


In [3]:
contract = {split: {
    "scores": f"predictions/{split}/submission_article_scores.parquet",
    "rankings": f"rankings/{split}/submission_top_200.parquet",
    "recorded_slates": f"diagnostics/recorded_slates/{split}_scores.parquet",
    "request_manifest": f"diagnostics/catalogue/{split}_requests.parquet",
    "request_scope": SCOPES[f"export_{split}"]["info"], **EXPORTS[split]["summary"],
    "validation": EXPORTS[split]["validation"],
} for split in D.SPLITS}
common = {"export_schema_version": 5, "prediction_pool": C.PREDICTION_POOL,
          "selection_k": C.SELECTION_K, "export_k": C.EXPORT_K, "shortlist_size": C.SHORTLIST_SIZE,
          "final_display_size": C.FINAL_DISPLAY_SIZE, "score_semantics": "uncalibrated raw neural ranking score; higher is better",
          "pool_policy": ws.reports["pool_policy"], "pool_fingerprint": ws.reports["pool_fingerprint"],
          "eligibility_rules": C.ELIGIBILITY_RULES, "tie_order": "score descending, original article_id ascending"}
EXPORT_MANIFEST = {**common, "splits": contract, "all_training_and_export_requests": True,
                   "recorded_slates": RECORDED, "resource_estimate": BUDGET,
                   "labels": "validation labels joined separately; project test outcomes masked"}
METRICS = {"definitions": P.METRIC_DEFINITIONS, "selection": SELECTION, "selection_protocol": SELECTOR.protocol,
           "selection_validation": {**SELECTED_SAMPLE, "scope": SCOPES["selection_validation"]["info"]},
           "exported_validation": {**EXPORTED_VALIDATION, "scope": SCOPES["export_validation"]["info"]},
           "full_validation_catalogue": EXPORTED_VALIDATION,
           "full_validation_recorded_slates": VAL_DIAGNOSTIC,
           "full_validation_newest_first": NEWEST_FIRST_METRICS,
           "primary": "known_click_recall@100", "secondary": "observed_click_ndcg@100",
           "diagnostic_recorded_slates_validation": VAL_DIAGNOSTIC,
           "training_history": "training_history.json", "tuning": STUDY_SUMMARY,
           "test": "labelled project holdout; outcome evaluation deferred until full pipeline freeze",
           "limitations": ["Validation reused for selection: development results.",
                           "Original ineligible/missed clicks remain in recall denominators and ideal DCG.",
                           "Observed-click recovery is exposure-biased; unshown outcomes remain unknown.",
                           "Training predictions are in-sample, not out-of-fold predictions."]}
RUN_CONFIG = {**common, "experiment_id": P.EXPERIMENT_ID, "run_id": RUN_ID, "owner": OWNER, "seed": SEED,
              "code": CODE_SOURCE, "environment": environment_info(), "config": CFG,
              "dataset_split": SPLIT_MANIFEST, "data_root": str(DATA_ROOT), "resolved_paths": ws.paths.as_dict(), "input_manifest": "input_manifest.json",
              "feature_order": FEATURE_ORDER, "feature_audit": F.FEATURE_AUDIT,
              "preprocessing": "model/preprocessing.json", "feature_dictionary": "model/feature_dictionary.json",
              "article_id_order": "model/article_ids.npy", "model_config": MODEL_CFG.to_json(), "parameters": PARAMS,
              "training_scope": TRAIN_SUBSET, "chosen_checkpoint": SELECTION["selected_checkpoint"],
              "selection_protocol": SELECTOR.protocol, "request_scopes": {key: value["info"] for key, value in SCOPES.items()},
              "request_scope_ids": "selection/request_scopes.json", "coverage": PREFLIGHT["coverage"],
              "recorded_ineligible": PREFLIGHT["recorded_ineligible"], "embedding_sources": ws.vector_reports,
              "history": ws.reports["history"], "export_counts": {split: result["summary"] for split, result in EXPORTS.items()},
              "tuning_objective": STUDY_SUMMARY["objective"], "training_objective": "BCE on supplied recorded pairs only",
              "persistent_location": str(PERSIST_DIR)}
for name, value in (("export_manifest.json", EXPORT_MANIFEST), ("metrics.json", METRICS), ("run_config.json", RUN_CONFIG)):
    atomic_write_json(RUN_DIR / name, value)
(RUN_DIR / "requirements.txt").write_text(P.requirements_text())
with log.stage("save complete run to shared Drive"):
    P.persist_run(RUN_DIR, PERSIST_DIR, log=log)
print("Persistent run:", PERSIST_DIR)


2026-11-04 09:19:38 [start] save complete run to shared Drive
2026-11-04 09:19:50 persisted 62 files to /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/20261104_014752_seed42
2026-11-04 09:19:50 [done] save complete run to shared Drive (11.2s)
Persistent run: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/20261104_014752_seed42


## 8. Reload check

Load the selected weights and fitted preprocessing from persistent storage and
compare 128 recorded-candidate scores with the saved predictions. The tolerance
is `1e-4`; article IDs and named feature mappings are retained in the bundle.


In [4]:
from huim.persist import load_model as reload_model
loaded, loaded_cfg, loaded_pre, loaded_meta = reload_model(PERSIST_DIR, DEVICE)
assert loaded_pre.to_json() == ws.pre.to_json()
ctx = ws.ctx["validation"].to(DEVICE)
art = ws.art.to(DEVICE)
rec = ctx.recorded
positions = torch.as_tensor(rec["req_pos"][:128], device=DEVICE)
articles = torch.as_tensor(rec["art_idx"][:128], device=DEVICE)
with torch.inference_mode():
    raw = loaded(ctx.tensors, art.tensors, len(art.ids), positions, articles).float()
    age = (ctx.tensors["req_s"][positions]-art.tensors["pub_s"][articles]).double()/3600.
    age = age.clamp_min(0).masked_fill(art.tensors["pub_missing"][articles], 0.)
    weight = 0.0
    scores = (raw-weight*torch.log1p(age).float()).cpu().numpy()
probe = pd.DataFrame({"behaviour_id": ctx.requests.behaviour_id.to_numpy()[rec["req_pos"][:128]],
    "article_id": rec["article_id"][:128], "loaded_score": scores})
reference = pd.read_parquet(PERSIST_DIR/"diagnostics/recorded_slates/validation_scores.parquet")
matched = probe.merge(reference, on=["behaviour_id","article_id"], validate="one_to_one")
delta = float(np.abs(matched.loaded_score-matched.score).max())
RELOAD = {"recorded_pairs": len(matched), "max_abs_difference": delta,
          "tolerance": 1e-4, "passed": bool(len(matched)==128 and delta<=1e-4)}
atomic_write_json(RUN_DIR/"reload_check.json", RELOAD)
shutil.copy2(RUN_DIR/"reload_check.json", PERSIST_DIR/"reload_check.json")
assert RELOAD["passed"], RELOAD
print("Numerical reload check:", RELOAD)


Numerical reload check: {'recorded_pairs': 128, 'max_abs_difference': 1.430511474609375e-06, 'tolerance': 0.0001, 'passed': True}


## 9. Business interpretation and handoff

### Completed run and validation findings

Run `20261104_014752_seed42` completed 12 Optuna trials (six complete, six
pruned), followed by four epochs on all 232,887 training requests. The selected
checkpoint was the end of the third epoch (`epoch=2`, `global_step=1896`),
rather than the final weights. The model has 114,462 trainable parameters.

| Measure | 100-request model-selection record | SMALL evaluation: 29,110 requests |
| --- | ---: | ---: |
| Macro known-click Recall@100 | 0.8250 | 0.7915 |
| Observed-click nDCG@100 | 0.2469 | 0.2209 |
| Mean recorded-slate AUC | — | 0.7175 |

Across all validation requests, the top 100 recovered an average **79.15% of
each request's known clicks** from the 20,738-article SMALL catalogue. All 29,249 observed clicked articles were eligible,
and no request had an empty catalogue pool, so the missed clicks were not caused
by publication-cutoff exclusions. The gap from 82.50% on the small selection
sample shows why the full-validation result is the appropriate development
summary. It does not establish the cause of that gap or an unbiased test result:
validation also informed tuning and checkpoint selection.

Newest-first achieved 0.9074 Recall@100 and 0.2506 nDCG@100 on the same
requests. The raw HUIM score therefore does not outperform explicit recency;
this motivates evaluating the separate freshness blend directly.

The attention mechanism allows different candidates to match different past
interests without a recurrent sequence encoder. This is a design rationale,
not an isolated finding: no HUIM component ablations or repeated-seed comparisons
were performed. The results therefore support this fitted model as a shortlist
candidate, but do not establish that attention outperforms a history average or
that it is the strongest model in the project.

### Business meaning and limitations

Recovering previously clicked articles keeps evidence of reader interest
available to the business reranker. The working relevance cutoff remains 100;
the additional 100 exported articles are a buffer for later analysis. Stage 2
must preserve relevance while applying the agreed business priorities. AUC
describes discrimination among originally shown candidates, while Recall@100
describes recovery from the much larger eligible catalogue; they answer
different questions.

These offline results do not measure higher online click-through rate, reader
retention or profit. Unshown articles have unknown outcomes, and observed-click
retrieval remains affected by the original exposure policy. The sigmoid of the
model logit is uncalibrated and should not be interpreted as catalogue CTR.
Vector availability and source-snapshot limitations also remain relevant when
interpreting recommendations.

### Completed handoff and remaining analysis

The export covered **232,887 training, 29,110 validation and 29,142 holdout
requests**, retaining 200 genuine catalogue-ranked articles per request. Validation and holdout scoring/writing intervals were 118.4 and 113.8 seconds,
respectively, before subsequent verification and Drive copying. They scored
566,218,581 and 566,598,922 eligible pairs. Final neural fitting took 203.6
seconds. Training predictions are in-sample and are not independent evaluation
evidence.

The persistent run contains the rankings and scores, complete recorded-slate
diagnostics, run configuration, selected weights, fitted preprocessing, feature
names/category maps, original article-ID order, helper code and tuning history.
The saved-model check passed on 128 recorded candidate scores. Its maximum
absolute difference was `1.43e-6`, below the declared `1e-4` tolerance. The saved
preprocessing and article-ID lookups preserve the meaning and order of inputs.

Further evaluation can use these saved validation artifacts for shared model
comparisons, user-group performance and recommendation examples without another
full export. Later explainability can reload the model and its named inputs for
a small, declared set of cases; attention weights alone are not causal
explanations. No further HUIM retraining or component-ablation run is included
in this experiment. Holdout outcome evaluation remains deferred until the whole
pipeline is frozen, and reranker tuning uses validation only.


In [5]:
print("Selected validation sample:", {key: SELECTED_SAMPLE[key] for key in
                                      ("known_click_recall@100", "observed_click_ndcg@100")})
print("All validation requests:", {key: EXPORTED_VALIDATION[key] for key in
                                  ("known_click_recall@100", "observed_click_ndcg@100")})
print("Actual tuning outcomes:", STUDY_SUMMARY["actual_trials"], STUDY_SUMMARY["states"])
print("Selected checkpoint:", SELECTION["selected_checkpoint"], "Stop reason:", TRAIN_STATE["stop_reason"])
print("Persistent reload passed:", RELOAD["passed"])
print("Handoff folder:", PERSIST_DIR)


Selected validation sample: {'known_click_recall@100': 0.825, 'observed_click_ndcg@100': 0.2469184196729059}
All validation requests: {'known_click_recall@100': 0.7914886064353601, 'observed_click_ndcg@100': 0.2208721165463811}
Actual tuning outcomes: 12 {'COMPLETE': 6, 'PRUNED': 6}
Selected checkpoint: {'eval_index': 8, 'epoch': 2, 'step_in_epoch': 632, 'global_step': 1896} Stop reason: max_epochs=4 reached
Persistent reload passed: True
Handoff folder: /content/drive/MyDrive/Colab Notebooks/BT4222 Shared Final Folder/Experiments/2.2_huim/20261104_014752_seed42
